In [11]:
from langgraph.graph import START,END,StateGraph
from typing import TypedDict,Annotated
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.graph.message import add_messages
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode,tools_condition
from llm import llm
import cmath

In [12]:
@tool
def is_prime(a:int)->str:
    """Check if a number is prime or not"""
    if(a<2):
        return 'no'
    for i in range(2,a):
        if(a%i==0):
            return 'no'
    return 'yes'
@tool
def sqrt(a:int)->float:
    """Calculate the square root of a number"""
    return cmath.sqrt(a)
@tool
def is_evenodd(a:int)->str:
    """Check is number is even or odd"""
    if(a%2==0):
        return "Even"
    return "Odd"
@tool
def get_factorial(a:int):
    """Calculate factorial of a number""" 
    if(a>10):
        return "Too large"
    if(a==0):
        return 1
    if(a<0):
        return "Does not exist"
    fact=1
    for i in range(1,a+1):
        fact*=i
    return fact 
@tool
def fun_fact(a:int)->str:
    """A fun fact about the number"""
    return "Fact - its a number :)"


In [13]:
tools=[sqrt,is_evenodd,get_factorial,fun_fact]
toolnode=ToolNode(tools)
LLM=llm.bind_tools(tools)

In [14]:
class NumState(TypedDict):
    num:int
    messages:Annotated[list[BaseMessage],add_messages]

In [15]:
def brain(state:NumState):
    res=LLM.invoke(state['messages'])
    print(res)
    return {'messages':[res]}

In [16]:
graph=StateGraph(NumState)

graph.add_node("brain",brain)
graph.add_node("tools",toolnode)

graph.add_edge(START,"brain")
graph.add_conditional_edges("brain",tools_condition)
graph.add_edge("tools",'brain')

workflow=graph.compile()


In [17]:
res=workflow.invoke({'messages':[HumanMessage(content="analysis of 7")]})

content='' additional_kwargs={'tool_calls': [{'id': 'a9v5q8y3a', 'function': {'arguments': '{"a":7}', 'name': 'is_evenodd'}, 'type': 'function'}, {'id': 'yantsf8za', 'function': {'arguments': '{"a":7}', 'name': 'sqrt'}, 'type': 'function'}, {'id': '6qkpdq75c', 'function': {'arguments': '{"a":7}', 'name': 'get_factorial'}, 'type': 'function'}, {'id': 'pe8976fnt', 'function': {'arguments': '{"a":7}', 'name': 'fun_fact'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 373, 'total_tokens': 425, 'completion_time': 0.101115265, 'completion_tokens_details': None, 'prompt_time': 0.022996349, 'prompt_tokens_details': None, 'queue_time': 0.34754003, 'total_time': 0.124111614}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e3167-b4ad-7351-aa68-3bdd66ec97bb-0' tool_calls=[{'name': 'is_evenodd', '

In [18]:
res['messages'][-1].text


"The number 7 is odd, its square root is approximately 2.65, its factorial is 5040, and a fun fact about the number 7 is that it's a number."

In [19]:
res

{'messages': [HumanMessage(content='analysis of 7', additional_kwargs={}, response_metadata={}, id='3c15cbf8-2d3a-472d-acc6-a9832ad8e87b'),
  AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'a9v5q8y3a', 'function': {'arguments': '{"a":7}', 'name': 'is_evenodd'}, 'type': 'function'}, {'id': 'yantsf8za', 'function': {'arguments': '{"a":7}', 'name': 'sqrt'}, 'type': 'function'}, {'id': '6qkpdq75c', 'function': {'arguments': '{"a":7}', 'name': 'get_factorial'}, 'type': 'function'}, {'id': 'pe8976fnt', 'function': {'arguments': '{"a":7}', 'name': 'fun_fact'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 52, 'prompt_tokens': 373, 'total_tokens': 425, 'completion_time': 0.101115265, 'completion_tokens_details': None, 'prompt_time': 0.022996349, 'prompt_tokens_details': None, 'queue_time': 0.34754003, 'total_time': 0.124111614}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_f8b414701e', 'service_tier': 'on_demand', 'finish_r